# Function Pointers

---

Function pointers are another type in C that allows us to address the executable code of a function by their location in memory (typically the first instruction in the function call).

They are useful for reducing code size and for constructing conventional programming patterns. For example:
- Providing a sorting algorithm on objects with a method to compare two elements.
- Giving the option to change implementation choices at run-time.

To declare a function pointer, declare its return type, put its name in `(*...)`, and give only the types of the arguments that it will accept. As the function signature is part of the type information, the compiler will know when an appropriate function has been assigned.
```c
int (*fptr) (int, int);
```
So if we have a function `int func(int x, int y)`, we can do
```c
fptr = func;
```
Note that putting the `&` operator in front of `func` is optional.

Now `fptr` refers to a function and can execute it.
```c
fptr(1, 2);
```
Sidenote: we can also use a typedef on a function pointer.
```c
typedef int (*fptr_t) (int, int);
```

---
---

# Signals

---

## Sending

Processes communicate with one another using signals, which are a type of software interrupt. 

On receiving a signal, the process will have its execution interrupted to call a user specified function. Upon returning from this function, execution is resumed.

The operating system will generate signals, such as segmentation faults when a user attempts to access out of bounds memory.

For user-generated signals, the `kill` system call can be used (requires `<sys/types.h>` and `<signal.h>`):
```c
int kill (pid_t pid, int sig);
```

Some signals you may be familiar with:
- SIGINT - pressing Ctrl+c in your shell
- SIGSTOP - pressing Ctrl+z in your shell
- SIGTERM - asks the process to terminate
- SIGKILL - forces the process to terminate

All signals apart from SIGKILL and SIGSTOP can be handled by the process by defining a signal handler.

In [1]:
%man 7 signal | head -n25

signal(7)		Miscellaneous Information Manual	      signal(7)

NAME
       signal - overview of signals

DESCRIPTION
       Linux  supports	both POSIX reliable signals (hereinafter "standard sig‐
       nals") and POSIX real-time signals.

   Signal dispositions
       Each signal has a current disposition, which determines how the	process
       behaves when it is delivered the signal.

       The  entries  in	 the "Action" column of the table below specify the de‐
       fault disposition for each signal, as follows:

       Term   Default action is to terminate the process.

       Ign    Default action is to ignore the signal.

       Core   Default action is to terminate the process  and  dump  core  (see
	      core(5)).

       Stop   Default action is to stop the process.



---

## Catching

We can setup a custom signal handler that will trigger the provided function when interrupted by certain signals.
```c
void sigint_handler(int signo, siginfo_t* sinfo, void* context);
```
```c
struct sigaction sig = {0};

sig.sa_sigaction = sigint_handler; // specifies the function
sig.sa_flags = SA_SIGINFO;

sigaction(SIGINT, &sig, NULL); // specifies the signal and action
```
The signal handler should not be too long, or otherwise we'll miss out on other signals while executing the handler.

---

## Safety

Many common I/O functions like `printf`, and memory management functions like `malloc`, are not safe to use inside a signal handling function. This is because these functions often have complex intenal logic, which may take a non-trivial amount of time to complete.

If these functions were to be used in a signal handler:
- If another signal arrives while the handler is running, it cannot be handled straight away.
- If the execution of one of these functions is interrupted, then the program risks interacting with stale or inconsistent data that was not fully updated from the previous function call.

Therefore, only use async-signal-safe functions inside a handler.

In [1]:
%man signal-safety | head -n35

signal-safety(7)	Miscellaneous Information Manual       signal-safety(7)

NAME
       signal-safety - async-signal-safe functions

DESCRIPTION
       An  async-signal-safe  function	is  one	 that can be safely called from
       within a signal handler.	 Many functions are not async-signal-safe.   In
       particular,  nonreentrant  functions are generally unsafe to call from a
       signal handler.

       The kinds of issues that render a function unsafe can be quickly	 under‐
       stood when one considers the implementation of the stdio library, all of
       whose functions are not async-signal-safe.

       When  performing	 buffered I/O on a file, the stdio functions must main‐
       tain a statically allocated data buffer along with  associated  counters
       and indexes (or pointers) that record the amount of data and the current
       position	 in the buffer.	 Suppose that the main program is in the middle
       of a call to a stdio function such as printf(3) where th

---

## Pause

This is a blocking function that instructs the current process to suspend execution until it receives a signal:
```c
int pause(void);
```

---

## errno

Most C functions will report errors via return values, but will additionally use the global variable `errno` found in `<errno.h>`.

Failed system calls typically set `errno` to an integer unique to the error type.
```c
errno = 0;
FILE* fp = fopen("fakefile.txt", "r");
if (errno != 0) perror("Error occurred");
else fclose(fp);
```
Use `strerror` to get the string descriptor or `perror` to print the error message of the errno code to stderr.

Remember to always reset `errno = 0` before/after any error checks. 

---
---

# Low-Level I/O

---

## Device I/O

Everything in UNIX is either a file or a process. Processes each have a file descriptor table set aside for keeping track of what files they have open.

The file descriptor is a simple numerical identifier. 0, 1 and 2 are typically reserved for `stdin`, `stdout` and `stderr` respectively. Any other subsequently opened file is the next largest value (3, 4, so on).

![file descriptor](img/file-descriptor.png)

---

## Low-Level I/O Functions

If we want full control over the I/O device, we can use the `<fcntl.h>` and `<unistd.h>` functions. Using them, we can specify the I/O modes such as blocking/non-blocking, no/full/line buffering, etc.

To open a file with a system call, use `open`. This will return a file descriptor.

The `read` and `write` system calls gives us the most primitive input/output alternatives, which operate on file descriptors.

Remember to `close`.

### Binary Stream I/O

Operating on a `FILE*` object uses the `fread` and `fwrite` abstractions, which utilise binary stream input and output functions. 

Since it is a file stream this may mean contents written by `fwrite` might not have been flushed. To immediately write to disk, `fflush`, and if you have finished `fclose`.

---

## Blocking and Non-Blocking

Blocking functions will wait until the I/O action has been fully completed before proceeding to the next line of code.
- `scanf`/`fgets` on stdin waits until you type some input and press enter
- `read` blocks if the buffer is empty
- `write` blocks if the buffer is full

Non-blocking functions  will not wait for the I/O operation to be fully complete and will immediately grab the data it can before proceeding to the next line of execution.

Typically non-blocking is used in conjunction with interrupts, where upon receiving a signal will know that data has arrived and will handle it. 